# Tensor Engineering for HCP Data

## Mathematical Logic of Tensor Transformation
The input data is in a flat, longitudinal format where each row represents a single week of data for a specific Healthcare Professional (HCP). Since there are 86 weeks of data, each HCP corresponds to 86 rows.

To apply Deep Sequence Learning (like 1D-CNNs or GRUs), we need to project this 2D representation into a 3D Tensor $\mathcal{T} \in \mathbb{R}^{N \times S \times F}$ where:
- $N$ = Number of distinct HCPs
- $S$ = Sequence length (86 weeks strictly ordered by `WEEK_IDX`)
- $F$ = Number of numerical features

The target $\mathcal{Y} \in \mathbb{R}^N$ and fold assignments $\mathcal{K} \in \mathbb{R}^N$ will be strictly aligned with the order of HCPs in the tensor.

In [30]:
import pandas as pd
import numpy as np
import torch
import os
from pathlib import Path
import gc

# Paths
BASE_DIR = Path("../../").resolve()
DATA_DIR = BASE_DIR / "data"
SILVER_DATA_PATH = DATA_DIR / "silver_layer_longitudinal.parquet"
HCP_MANIFEST_PATH = DATA_DIR / "hcp_manifest.parquet"
TENSORS_OUT_DIR = BASE_DIR / "/Users/davidbazalduamendez/Documents/GitHub/Pfizer-segmentation-Ulcerative-Colitis/models/1d-CNN/tensors"

# Fallbacks if paths differ
if not SILVER_DATA_PATH.exists():
    SILVER_DATA_PATH = DATA_DIR / "silver_layer_longitudinal.parquet"
    HCP_MANIFEST_PATH = DATA_DIR / "hcp_manifest.parquet"

TENSORS_OUT_DIR.mkdir(parents=True, exist_ok=True)

In [31]:
print("Loading Silver Data...")
df_long = pd.read_parquet(SILVER_DATA_PATH)
df_manifest = pd.read_parquet(HCP_MANIFEST_PATH)

print(f"Longitudinal Data Shape: {df_long.shape}")
print(f"Manifest Data Shape: {df_manifest.shape}")

Loading Silver Data...
Longitudinal Data Shape: (1800066, 76)
Manifest Data Shape: (20931, 11)


In [32]:
# Feature Selection
ignore_cols = ['NUEVO_ID', 'WEEK_IDX', 'HCP_FOLD', 'ATSEG',"IS_LABELED_HCP"]
numeric_cols = df_long.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in ignore_cols]

print(f"Selected {len(feature_cols)} features for sequence modeling.")

Selected 65 features for sequence modeling.


In [33]:
from sklearn.preprocessing import StandardScaler

print("Sorting data to ensure strict temporal order...")
df_long = df_long.sort_values(by=['NUEVO_ID', 'WEEK_IDX']).reset_index(drop=True)

# Identify labeled HCPs to fit scaler without Data Leakage
# Labeled HCPs shouldn't be null or 'UNLABELED'
if 'ATSEG' in df_manifest.columns:
    labeled_hcps = df_manifest[df_manifest['ATSEG'].notna() & (df_manifest['ATSEG'] != 'UNLABELED') & (df_manifest['ATSEG'] != -1)]['NUEVO_ID'].unique()
else:
    labeled_hcps = df_manifest['NUEVO_ID'].unique()

print(f"Number of Labeled HCPs for scaling fit: {len(labeled_hcps)}")

# Filter df_long to only labeled HCPs
df_long_labeled = df_long[df_long['NUEVO_ID'].isin(labeled_hcps)]

# Fit StandardScaler strictly on labeled data to prevent Data Leakage
scaler = StandardScaler()
print("Fitting StandardScaler on labeled data...")
scaler.fit(df_long_labeled[feature_cols])

# Transform all data
print("Transforming all features...")
features_scaled = scaler.transform(df_long[feature_cols])

# 3D Tensor Construction
print("Reshaping to 3D Tensor...")
num_hcps = df_long['NUEVO_ID'].nunique()
num_features = len(feature_cols)
seq_length = 86

assert len(df_long) == num_hcps * seq_length, "Data length mismatch! Some HCPs might not have exactly 86 weeks."

# Efficient reshape
tensor_3d = features_scaled.reshape(num_hcps, seq_length, num_features)

# Extract unique HCPs in the exact same sorted order
unique_hcps = df_long['NUEVO_ID'].unique()


Sorting data to ensure strict temporal order...
Number of Labeled HCPs for scaling fit: 20931
Fitting StandardScaler on labeled data...
Transforming all features...
Reshaping to 3D Tensor...


In [34]:
print("Aligning labels and folds...")
# Ensure manifest is aligned with the tensor order
df_manifest_aligned = pd.DataFrame({'NUEVO_ID': unique_hcps})
df_manifest_aligned = df_manifest_aligned.merge(df_manifest, on='NUEVO_ID', how='left')

# Encode ATSEG string categories into numerical integers (0, 1, 2) if necessary
if df_manifest_aligned['ATSEG_HCP'].dtype == 'O' or str(df_manifest_aligned['ATSEG_HCP'].dtype) == 'category':
    df_manifest_aligned['ATSEG_encoded'], unique_classes = pd.factorize(df_manifest_aligned['ATSEG_HCP'])
    print(f"Encoded ATSEG classes: {unique_classes.tolist()}")
    labels = df_manifest_aligned['ATSEG_encoded'].values
else:
    labels = df_manifest_aligned['ATSEG_HCP'].values

# Ensure no NaNs and cast to int
labels = pd.Series(labels).fillna(-1).astype(int).values
folds = df_manifest_aligned['HCP_FOLD'].fillna(-1).astype(int).values

# Convert to PyTorch Tensors
X_tensor = torch.tensor(tensor_3d, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)
fold_tensor = torch.tensor(folds, dtype=torch.long)

print(f"X Tensor Shape: {X_tensor.shape}")
print(f"y Tensor Shape: {y_tensor.shape}")


Aligning labels and folds...
Encoded ATSEG classes: ['SEG_A', 'SEG_B', 'SEG_C']
X Tensor Shape: torch.Size([20931, 86, 65])
y Tensor Shape: torch.Size([20931])


In [36]:
print("Serializing to disk...")
torch.save(X_tensor, TENSORS_OUT_DIR / "X_features.pt")
torch.save(y_tensor, TENSORS_OUT_DIR / "y_labels.pt")
torch.save(fold_tensor, TENSORS_OUT_DIR / "folds.pt")

print(f"Successfully saved tensors to {TENSORS_OUT_DIR}")


Serializing to disk...
Successfully saved tensors to /Users/davidbazalduamendez/Documents/GitHub/Pfizer-segmentation-Ulcerative-Colitis/models/1d-CNN/tensors
